# Handling HTTP status codes

The server is a separate process, `server.py` in this folder. Start it from the repo root, in a terminal:

```
uvicorn MLAppDev.ModB.Sprint1.server:app --reload --port 8010
```

Keep that terminal open. Every block below sends one request, prints the status, and then shows what the client does about it.

Run the blocks in order the first time. After that, go back and change the arguments, that is the point of having them in cells.

In [7]:
import time

import requests

BASE_URL = "http://127.0.0.1:8010"
TIMEOUT = 60  # seconds, requests waits forever if you do not set one

In [8]:
def show(label, response):
    print(f"\n{label}")
    print(f"  status: {response.status_code} {response.reason}")
    try:
        print(f"  body:   {response.json()}")
    except ValueError:
        print(f"  body:   {response.text!r}")  # not every response has a JSON body

## 1. 200 OK and 404 Not Found

The plain way: read `status_code` and decide. 2xx means the body is what you asked for, 4xx means the request is wrong and retrying will not help.

In [36]:
def get_item(item_id):
    response = requests.get(f"{BASE_URL}/items/{item_id}", timeout=TIMEOUT)
    show(f"GET /items/{item_id}", response)
    if response.status_code == 404:
        print(f"  handling: 4xx, do not retry. detail = {response.json()['detail']}")
        return
    print(f"  handling: 2xx, use the body: {response.json()['name']}")


get_item(1)   # 200, the item exists
get_item(99)  # 404, it does not


GET /items/1
  status: 200 OK
  body:   {'item_id': 1, 'name': 'keyboard', 'price': 40.0}
  handling: 2xx, use the body: keyboard

GET /items/99
  status: 404 Not Found
  body:   {'detail': 'item 99 not found'}
  handling: 4xx, do not retry. detail = item 99 not found


## 2. The same thing with `raise_for_status()`

The other style: turn every 4xx and 5xx into an exception and let one `try/except` deal with it.

In [37]:
response = requests.get(f"{BASE_URL}/items/99", timeout=TIMEOUT)
print("GET /items/99 with raise_for_status")
try:
    response.raise_for_status()
except requests.HTTPError as error:
    print(f"  handling: raised {error}")
    print(f"  detail:   {response.json()['detail']}")

GET /items/99 with raise_for_status
  handling: raised 404 Client Error: Not Found for url: http://127.0.0.1:8010/items/99
  detail:   item 99 not found


## 3. Success is not only 200

- **201 Created** means the item now exists, and the `Location` header points to it.
- **202 Accepted** means we took the job but it is not finished. 200 would be a lie.

In [41]:
# 201
response = requests.post(
    f"{BASE_URL}/items", json={"name": "monitor", "price": 150}, timeout=TIMEOUT
)
show("POST /items", response)
print(f"  handling: 201, the item exists at {response.headers['Location']}")



POST /items
  status: 201 Created
  body:   {'item_id': 3, 'name': 'monitor', 'price': 150.0}
  handling: 201, the item exists at /items/3


In [42]:

# 202
response = requests.post(f"{BASE_URL}/jobs", timeout=TIMEOUT)
show("POST /jobs", response)
print("  handling: 202, it is queued only. Poll later, do not call it done.")



POST /jobs
  status: 202 Accepted
  body:   {'job_id': 42, 'status': 'queued'}
  handling: 202, it is queued only. Poll later, do not call it done.


## 4. 422 Unprocessable Entity

Nothing to write in the server. FastAPI checks the body against the `Item` model and answers 422 by itself. Try sending `"price": "cheap"` instead of removing it.

In [45]:
response = requests.post(f"{BASE_URL}/items", json={"name": "monitor"}, timeout=TIMEOUT)
show("POST /items with price missing", response)
for error in response.json()["detail"]:
    print(f"  handling: field {error['loc']} -> {error['msg']}")


POST /items with price missing
  status: 422 Unprocessable Entity
  body:   {'detail': [{'type': 'missing', 'loc': ['body', 'price'], 'msg': 'Field required', 'input': {'name': 'monitor'}}]}
  handling: field ['body', 'price'] -> Field required


## 5. 401 Unauthorized and 403 Forbidden

They look similar and they are not. 401 means "I do not know who you are" and the server tells you how to log in. 403 means "I know who you are, and you are not allowed".

In [53]:
def admin(token=None):
    params = {"token": token} if token else None
    response = requests.get(f"{BASE_URL}/admin", params=params, timeout=TIMEOUT)
    show(f"GET /admin token={token}", response)
    if response.status_code == 401:
        print(f"  handling: 401, we sent no credentials. Server says use {response.headers['WWW-Authenticate']}")
    elif response.status_code == 403:
        print("  handling: 403, the credentials were fine, this user is just not allowed")
    else:
        print(f"  handling: 200, {response.json()['message']}")


admin("admin")      # 401, no token at all



GET /admin token=admin
  status: 200 OK
  body:   {'message': 'welcome, admin'}
  handling: 200, welcome, admin


In [ ]:

admin("ana")    # 403, a token, but not an admin
admin("admin")  # 200

## 6. 400 Bad Request and 409 Conflict

400 is for input that breaks a business rule, even when the types are fine. 409 is for a valid request that clashes with the current state, like an item with no stock.

In [8]:
response = requests.get(f"{BASE_URL}/divide", params={"a": 10, "b": 0}, timeout=TIMEOUT)
show("GET /divide?b=0", response)
print(f"  handling: 400, fix the input: {response.json()['detail']}")


GET /divide?b=0
  status: 400 Bad Request
  body:   {'detail': 'b cannot be zero'}
  handling: 400, fix the input: b cannot be zero


In [9]:
response = requests.post(f"{BASE_URL}/orders", params={"item_name": "keyboard"}, timeout=TIMEOUT)
show("POST /orders?item_name=keyboard", response)
print(f"  handling: 409, the request is fine, the current state is not: {response.json()['detail']}")

response = requests.post(f"{BASE_URL}/orders", params={"item_name": "mouse"}, timeout=TIMEOUT)
show("POST /orders?item_name=mouse", response)
print(f"  handling: {response.status_code}, there was stock, so it worked")


POST /orders?item_name=keyboard
  status: 409 Conflict
  body:   {'detail': 'keyboard is out of stock'}
  handling: 409, the request is fine, the current state is not: keyboard is out of stock

POST /orders?item_name=mouse
  status: 200 OK
  body:   {'item_name': 'mouse', 'left': 4}
  handling: 200, there was stock, so it worked


## 7. 429 Too Many Requests

The server says how long to wait in the `Retry-After` header. The polite client waits that long and tries again instead of hammering.

In [10]:
for attempt in range(1, 6):
    response = requests.get(f"{BASE_URL}/limited", timeout=TIMEOUT)
    print(f"\nGET /limited attempt {attempt}: {response.status_code} {response.json()}")
    if response.status_code != 429:
        continue
    wait = int(response.headers.get("Retry-After", "1"))
    print(f"  handling: 429, Retry-After says {wait}s, so we sleep and try once more")
    time.sleep(wait)


GET /limited attempt 1: 200 {'call': 1, 'limit': 3}

GET /limited attempt 2: 200 {'call': 2, 'limit': 3}

GET /limited attempt 3: 200 {'call': 3, 'limit': 3}

GET /limited attempt 4: 429 {'detail': 'more than 3 calls in 2.0s'}
  handling: 429, Retry-After says 2s, so we sleep and try once more



GET /limited attempt 5: 200 {'call': 1, 'limit': 3}


## 8. 5xx: the server is broken, not the request

Same request, different meaning. 500 is a bug, retrying the identical call changes nothing. 503 and 504 are temporary, so a retry with a pause is reasonable.

In [11]:
response = requests.get(f"{BASE_URL}/boom", timeout=TIMEOUT)
show("GET /boom", response)
print("  handling: 500, that is our bug, retrying will not fix it")

response = requests.get(f"{BASE_URL}/upstream-slow", timeout=TIMEOUT)
show("GET /upstream-slow", response)
print("  handling: 504, an upstream is slow. Retry later or serve a fallback.")


GET /boom
  status: 500 Internal Server Error
  body:   'Internal Server Error'
  handling: 500, that is our bug, retrying will not fix it



GET /upstream-slow
  status: 504 Gateway Timeout
  body:   {'detail': 'upstream did not answer in time'}
  handling: 504, an upstream is slow. Retry later or serve a fallback.


In [12]:
# 503: a dependency is down. Temporary, so we try three times with a pause.
print("\nGET /dependency")
for attempt in range(1, 4):
    response = requests.get(f"{BASE_URL}/dependency", timeout=TIMEOUT)
    print(f"  attempt {attempt}: {response.status_code} {response.json()['detail']}")
    time.sleep(0.5 * attempt)

print("  handling: still 503 after 3 attempts, stop and tell the user to try later")


GET /dependency
  attempt 1: 503 dependency is down


  attempt 2: 503 dependency is down


  attempt 3: 503 dependency is down


  handling: still 503 after 3 attempts, stop and tell the user to try later


## 9. 402 Payment Required, with an error body we designed

`server.py` turns a domain error (`InsufficientFunds`) into a status with one exception handler. The body is ours, not FastAPI's `{"detail": ...}`, and that is allowed: pick a shape and keep it.

In [13]:
def checkout(price):
    response = requests.get(f"{BASE_URL}/checkout", params={"price": price}, timeout=TIMEOUT)
    show(f"GET /checkout?price={price}", response)
    if response.status_code == 402:
        body = response.json()
        print(f"  handling: 402, tell the user: balance {body['balance']} < price {body['price']}")


checkout(5)   # 200
checkout(50)  # 402


GET /checkout?price=5
  status: 200 OK
  body:   {'charged': 5.0, 'balance': 5.0}

GET /checkout?price=50
  status: 402 Payment Required
  body:   {'error': 'insufficient_funds', 'balance': 10.0, 'price': 50.0}
  handling: 402, tell the user: balance 10.0 < price 50.0


## 10. 405 Method Not Allowed, and the request that never arrives

405 is free: the path exists, the method does not. And not every failure has a status code at all, a dead server or a slow one never answers.

In [14]:
response = requests.post(f"{BASE_URL}/items/1", timeout=TIMEOUT)
show("POST /items/1 (only GET is defined)", response)
print(f"  handling: 405, use a method the API documents: {response.json()['detail']}")


POST /items/1 (only GET is defined)
  status: 405 Method Not Allowed
  body:   {'detail': 'Method Not Allowed'}
  handling: 405, use a method the API documents: Method Not Allowed


In [15]:
# Nothing is listening on port 8099, so no response and no status code
print("GET http://127.0.0.1:8099/")
try:
    requests.get("http://127.0.0.1:8099/", timeout=2)
except requests.ConnectionError:
    print("  handling: ConnectionError, no status code, the server is not running")
except requests.Timeout:
    print("  handling: Timeout, we stopped waiting, there is no status code either")

GET http://127.0.0.1:8099/
  handling: ConnectionError, no status code, the server is not running


## 11. OpenRouter: 200 when it works, 502 when the provider fails

`POST /ask` on our server calls OpenRouter with the key from the repo root `.env`. When OpenRouter fails, that is not our client's fault, so our server answers **502 Bad Gateway** and passes the upstream message on.

In [16]:
response = requests.post(
    f"{BASE_URL}/ask",
    json={"question": "Explain HTTP status 502 in one sentence."},
    timeout=TIMEOUT,
)
show("POST /ask", response)
print(f"  handling: {response.status_code}")
if response.status_code == 200:
    print(f"  answer: {response.json()['answer']}")


POST /ask
  status: 200 OK
  body:   {'model': 'google/gemini-2.5-flash-lite', 'answer': 'A 502 Bad Gateway error means that one server on the internet received an invalid response from another server.'}
  handling: 200
  answer: A 502 Bad Gateway error means that one server on the internet received an invalid response from another server.


In [17]:
# Same endpoint, a model that does not exist, so the provider says no
response = requests.post(
    f"{BASE_URL}/ask",
    json={"question": "hi", "model": "not/a-real-model"},
    timeout=TIMEOUT,
)
show("POST /ask with a model that does not exist", response)
print(f"  handling: 502, our provider said no: {response.json()['detail']}")


POST /ask with a model that does not exist
  status: 502 Bad Gateway
  body:   {'detail': 'OpenRouter answered 400: not/a-real-model is not a valid model ID'}
  handling: 502, our provider said no: OpenRouter answered 400: not/a-real-model is not a valid model ID


## What to try

- Send `{"name": "monitor", "price": "cheap"}` to `POST /items` and look at the 422.
- Call `get_item(3)` after section 3, the monitor you created is item 3.
- Run section 7 twice in a row and watch the rate limit window reset.
- Call `POST /orders?item_name=mouse` five times, then look at `/orders?item_name=keyboard`.
- Restart the server to reset the items, the stock and the call counter. They live in memory.